In [5]:
import numpy as np

import sys
sys.path.insert(1, '../../human_me/')
from human_me import preprocess

from human_me.preprocess import correct_inputs as ci
from human_me.io import load_metabolic_model

full model

In [4]:
# prebuild = '/data2/hratch/human_me/prebuild/'
# ci.correct_model(model = prebuild + 'recon2_2.xml')
# non_machinery, revised_genes = ci.correct_psim(psim_df = input_data_path + 'psim_me.h5', fill_na = 'select', 
#                             non_machinery = {'HGNC:4556':['m', 'c'], 'HGNC:9251': ['l'], 
#                                             'HGNC:32043': ['e', 'n']})
# print(revised_genes)
# from human_me.expression import build_me_model
# me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False)
# me_model.pickle('/data2/hratch/human_me/full_12_29_20.pickle')

toy model

In [5]:
# other = '/data2/hratch/human_me/other/'
# ci.correct_model(model = other + 'toy_model.xml')
# revised_genes = ci.correct_psim(psim_df = input_data_path + 'psim_me.h5', fill_na = 'select', 
#                                non_machinery = None)
# print(revised_genes)
# from human_me.expression import build_me_model
# toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
#                                             unmodeled_protein_frac = None)
# toy_me_model.pickle(other + 'toy_me_12_29_20.pickle')
# sln, stat, _ = toy_me_model.solve_lp(mu_val = 1e-9)

core model

In [3]:
other_path = '/data2/hratch/human_me/other/'
mem = False # minimial media
fn = '/data2/hratch/human_me/other/core' 
if mem:
    fn += '_mem'

    
m_model = load_metabolic_model(fn + '.xml')
cm_1, cm_2, cm_3 = ci.correct_model(model_file = fn + '.xml')
# revised_genes = ci.correct_psim(psim_df = input_data_path + 'psim_me.h5', fill_na = 'select', 
#                                non_machinery = None)

# from human_me.expression import build_me_model
# toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
#                                             unmodeled_protein_frac = None)
# toy_me_model.pickle(other + 'toy_me_12_29_20.pickle')
# sln, stat, _ = toy_me_model.solve_lp(mu_val = 1e-9)

../../human_me/human_me/preprocess/correct_inputs.py:128 UserWarning: OIVD1m contains redundant complexes according to GPR, editing GPR
../../human_me/human_me/preprocess/correct_inputs.py:128 UserWarning: OIVD2m contains redundant complexes according to GPR, editing GPR
../../human_me/human_me/preprocess/correct_inputs.py:128 UserWarning: OIVD3m contains redundant complexes according to GPR, editing GPR
../../human_me/human_me/preprocess/correct_inputs.py:128 UserWarning: PFK contains redundant complexes according to GPR, editing GPR
../../human_me/human_me/preprocess/correct_inputs.py:150 UserWarning: Your metabolic model contains genes with HGNC:HGNC:####, changing to HGNC:####


Check for the recon2.2 HGNC:HGNC error
Remove genes not participating in reactions


../../human_me/human_me/preprocess/correct_inputs.py:207 UserWarning: gpi_hs does not exist in model. Adding to compartment r via sink This allows gpi_hs to be in the model at no cost.
../../human_me/human_me/preprocess/correct_inputs.py:207 UserWarning: uacgam does not exist in model. Adding to compartment g via sink This allows uacgam to be in the model at no cost.
../../human_me/human_me/preprocess/correct_inputs.py:207 UserWarning: udpacgal does not exist in model. Adding to compartment g via sink This allows udpacgal to be in the model at no cost.


If you want, you can check that the correct_model did not change the solution

In [18]:
np.allclose(m_model.slim_optimize(), cm_1.slim_optimize(), cm_2.slim_optimize())

True

You can also check that the qMINOS solver used by the ME Model works the same as the cobrapy native glpk solver

In [32]:
from human_me.me_solver.solve_me import qminosSolver
from cobra.util.solver import linear_reaction_coefficients

objective = {r.id: coef for r, coef in linear_reaction_coefficients(m_model).items()}
qminos_sln,_,_ = qminosSolver().solve_lp(me_model = m_model, 
                                     mu_val = None, 
                                     objective = objective)

reaction_indeces = [m_model.reactions.index(r_id) for r_id in list(objective)]
np.allclose(m_model.slim_optimize(), qminos_sln[reaction_indeces])

Getting MINOS parameters...
Done in 0.281807 seconds with status 0


True

# Preprocessing

Throughout preprocessing, we want to make sure that the metabolic model remains feasible for growth. If the metabolic model is not feasible, the ME Model will not be.

First, check if the metabolic model you plan to input is feasible:

Next, check if the biomass objective is formulated correctly. See the Biomass Objective notes for details. In short, we expect 

If biomass is not formulated correctly, we strongly recommend using our correcting function:

If you corrected, check again that the model remains feasible:

# ME Model Biomass Mass Fraction Sanity Check

Once we have generated the ME Model, we can check that the biomass formation reactions were implemented correctly. To do so, we must check that the expected mass fractions match the input mass fractions.The expected mass fraction is calculated as the sum of the stoichiometric coefficient * molecular weight of all the metabolites used in forming that biomass component. If the expected mass fraction does not match the input mass fraction, something is wrong either with the input mass fractions or the input coefficients for the biomass formation reactions 

In [ ]:
from human_me.core.biomass import check_me_biomass
from human_me.utils.parameters import biomass_parameters
expected_mass_fraction = check_me_biomass(me_model)

In our case, we used the default mass fraction values as input:

In [ ]:
mass_fraction = biomass_parameters.mass_fraction
mass_fraction

Finally, we can check if the expected mass fraction matched the input one (values should be ~0):

In [ ]:
difference = dict()
for biomass_type, mf in expected_mass_fraction.items():
    difference[biomass_type] = abs(mf - mass_fraction[biomass_type])
difference